In [0]:
display(dbutils.fs.ls("/Volumes/workspace/project1_dw/warehouse"))


path,name,size,modificationTime
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_airport.csv,dim_airport.csv,952963,1775805696000
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_country.csv,dim_country.csv,6341,1775805694000
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_date.csv,dim_date.csv,13116,1775805694000
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_flight_status.csv,dim_flight_status.csv,82,1775805694000
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_passenger.csv,dim_passenger.csv,5068697,1775805699000
dbfs:/Volumes/workspace/project1_dw/warehouse/dim_region.csv,dim_region.csv,82972,1775805695000
dbfs:/Volumes/workspace/project1_dw/warehouse/fact_passenger_flight.csv,fact_passenger_flight.csv,3725023,1775805698000


In [0]:
# Cell 1: 读取 7 个 CSV
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.project1_dw")
spark.sql("USE workspace.project1_dw")

base = "/Volumes/workspace/project1_dw/warehouse"

dim_date_raw = spark.read.option("header", True).csv(f"{base}/dim_date.csv")
dim_country_raw = spark.read.option("header", True).csv(f"{base}/dim_country.csv")
dim_region_raw = spark.read.option("header", True).csv(f"{base}/dim_region.csv")
dim_airport_raw = spark.read.option("header", True).csv(f"{base}/dim_airport.csv")
dim_passenger_raw = spark.read.option("header", True).csv(f"{base}/dim_passenger.csv")
dim_flight_status_raw = spark.read.option("header", True).csv(f"{base}/dim_flight_status.csv")
fact_passenger_flight_raw = spark.read.option("header", True).csv(f"{base}/fact_passenger_flight.csv")

for name, df in {
    "dim_date_raw": dim_date_raw,
    "dim_country_raw": dim_country_raw,
    "dim_region_raw": dim_region_raw,
    "dim_airport_raw": dim_airport_raw,
    "dim_passenger_raw": dim_passenger_raw,
    "dim_flight_status_raw": dim_flight_status_raw,
    "fact_passenger_flight_raw": fact_passenger_flight_raw
}.items():
    print(name, df.count())


dim_date_raw 365
dim_country_raw 236
dim_region_raw 2099
dim_airport_raw 9025
dim_passenger_raw 98620
dim_flight_status_raw 4
fact_passenger_flight_raw 98619


In [0]:
# Cell 2: 转换数据类型
dim_date = dim_date_raw.select(
    F.col("date_key").cast("int").alias("date_key"),
    F.col("full_date").cast("date").alias("full_date"),
    F.col("day_of_month").cast("int").alias("day_of_month"),
    F.col("month").cast("int").alias("month"),
    F.col("month_name").cast("string").alias("month_name"),
    F.col("quarter").cast("string").alias("quarter"),
    F.col("year").cast("int").alias("year")
)

dim_country = dim_country_raw.select(
    F.col("country_key").cast("int").alias("country_key"),
    F.col("country_code").cast("string").alias("country_code"),
    F.col("country_name").cast("string").alias("country_name"),
    F.col("continent_name").cast("string").alias("continent_name")
)

dim_region = dim_region_raw.select(
    F.col("region_key").cast("int").alias("region_key"),
    F.col("region_code").cast("string").alias("region_code"),
    F.col("region_name").cast("string").alias("region_name"),
    F.col("country_code").cast("string").alias("country_code"),
    F.col("continent_name").cast("string").alias("continent_name")
)

dim_airport = dim_airport_raw.select(
    F.col("airport_key").cast("int").alias("airport_key"),
    F.col("airport_ident").cast("string").alias("airport_ident"),
    F.col("airport_name").cast("string").alias("airport_name"),
    F.col("departure_airport_code").cast("string").alias("departure_airport_code"),
    F.col("airport_type").cast("string").alias("airport_type"),
    F.col("municipality").cast("string").alias("municipality"),
    F.col("scheduled_service").cast("string").alias("scheduled_service"),
    F.col("region_code").cast("string").alias("region_code"),
    F.col("country_code").cast("string").alias("country_code"),
    F.col("has_navaid").cast("string").alias("has_navaid"),
    F.col("navaid_count").cast("int").alias("navaid_count"),
    F.col("navaid_count_bucket").cast("string").alias("navaid_count_bucket"),
    F.col("navaid_type_group").cast("string").alias("navaid_type_group"),
    F.col("dominant_usage_type").cast("string").alias("dominant_usage_type"),
    F.col("max_power").cast("string").alias("max_power")
)

dim_passenger = dim_passenger_raw.select(
    F.col("passenger_key").cast("int").alias("passenger_key"),
    F.col("passenger_id").cast("string").alias("passenger_id"),
    F.col("first_name").cast("string").alias("first_name"),
    F.col("last_name").cast("string").alias("last_name"),
    F.col("gender").cast("string").alias("gender"),
    F.col("age").cast("int").alias("age"),
    F.col("age_group").cast("string").alias("age_group"),
    F.col("nationality").cast("string").alias("nationality")
)

dim_flight_status = dim_flight_status_raw.select(
    F.col("flight_status_key").cast("int").alias("flight_status_key"),
    F.col("flight_status").cast("string").alias("flight_status")
)

fact_passenger_flight = fact_passenger_flight_raw.select(
    F.col("fact_id").cast("int").alias("fact_id"),
    F.col("date_key").cast("int").alias("date_key"),
    F.col("airport_key").cast("int").alias("airport_key"),
    F.col("region_key").cast("int").alias("region_key"),
    F.col("country_key").cast("int").alias("country_key"),
    F.col("passenger_key").cast("int").alias("passenger_key"),
    F.col("flight_status_key").cast("int").alias("flight_status_key"),
    F.col("flight_record_count").cast("int").alias("flight_record_count"),
    F.col("delayed_flag").cast("int").alias("delayed_flag"),
    F.col("cancelled_flag").cast("int").alias("cancelled_flag"),
    F.col("ontime_flag").cast("int").alias("ontime_flag")
)


In [0]:
# cell3 检查数据
display(dim_date.limit(5))
display(dim_country.limit(5))
display(dim_region.limit(5))
display(dim_airport.limit(5))
display(dim_passenger.limit(5))
display(dim_flight_status.limit(5))
display(fact_passenger_flight.limit(5))


date_key,full_date,day_of_month,month,month_name,quarter,year
0,null,0,0,Unknown,null,0
1,2022-06-28,28,6,June,Q2,2022
2,2022-12-26,26,12,December,Q4,2022
3,2022-01-18,18,1,January,Q1,2022
4,2022-09-16,16,9,September,Q3,2022


country_key,country_code,country_name,continent_name
0,UNKNOWN,Unknown Country,Unknown
1,US,United States,North America
2,CA,Canada,North America
3,FR,France,Europe
4,BR,Brazil,South America


region_key,region_code,region_name,country_code,continent_name
0,UNKNOWN,Unknown Region,UNKNOWN,Unknown
1,US-AK,Alaska,US,North America
2,CA-NU,Nunavut,CA,North America
3,FR-ARA,Auvergne-Rhône-Alpes,FR,Europe
4,CA-QC,Quebec,CA,North America


airport_key,airport_ident,airport_name,departure_airport_code,airport_type,municipality,scheduled_service,region_code,country_code,has_navaid,navaid_count,navaid_count_bucket,navaid_type_group,dominant_usage_type,max_power
0,UNKNOWN,Unknown Airport,null,Unknown,Unknown,unknown,UNKNOWN,UNKNOWN,N,0,0,No Navaid,UNKNOWN,UNKNOWN
1,PACX,Coldfoot Airport,CXF,small_airport,Coldfoot,no,US-AK,US,N,0,0,No Navaid,UNKNOWN,UNKNOWN
2,CYCO,Kugluktuk Airport,YCO,small_airport,Kugluktuk,yes,CA-NU,CA,Y,1,1,NDB Family,BOTH,MEDIUM
3,LFLS,Grenoble Alpes Isère Airport,GNB,medium_airport,Grenoble,yes,FR-ARA,FR,N,0,0,No Navaid,UNKNOWN,UNKNOWN
4,CYND,Ottawa / Gatineau Airport,YND,medium_airport,Gatineau,yes,CA-QC,CA,N,0,0,No Navaid,UNKNOWN,UNKNOWN


passenger_key,passenger_id,first_name,last_name,gender,age,age_group,nationality
0,null,null,null,Unknown,null,Unknown,Unknown
1,ABVWIg,Edithe,Leggis,Female,62,60+,Japan
2,jkXXAX,Elwood,Catt,Male,62,60+,Nicaragua
3,CdUz2g,Darby,Felgate,Male,67,60+,Russia
4,BRS38V,Dominica,Pyle,Female,71,60+,China


flight_status_key,flight_status
0,Unknown
1,On Time
2,Delayed
3,Cancelled


fact_id,date_key,airport_key,region_key,country_key,passenger_key,flight_status_key,flight_record_count,delayed_flag,cancelled_flag,ontime_flag
1,1,1,1,1,1,1,1,0,0,1
2,2,2,2,2,2,1,1,0,0,1
3,3,3,3,3,3,1,1,0,0,1
4,4,4,4,2,4,2,1,1,0,0
5,5,5,5,1,5,1,1,0,0,1


In [0]:
# Cell 4: 保存为 Delta tables
dim_date.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_date")
dim_country.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_country")
dim_region.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_region")
dim_airport.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_airport")
dim_passenger.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_passenger")
dim_flight_status.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.dim_flight_status")
fact_passenger_flight.write.format("delta").mode("overwrite").saveAsTable("workspace.project1_dw.fact_passenger_flight")


In [0]:
# Cell 5: 验证表已经建好
for t in [
    "dim_date",
    "dim_country",
    "dim_region",
    "dim_airport",
    "dim_passenger",
    "dim_flight_status",
    "fact_passenger_flight"
]:
    print(t, spark.table(f"workspace.project1_dw.{t}").count())


dim_date 365
dim_country 236
dim_region 2099
dim_airport 9025
dim_passenger 98620
dim_flight_status 4
fact_passenger_flight 98619
